In [1]:
# Classificação Automática de Atividades Culturais e Educacionais
# 09 - Os classificadores
#
# Integrantes: Beatriz Aparecida de Mello Barbosa (10354067), Bruna Gonçalves Corte David (10425696),
#              Henrique Brainer Costa (10420717), João Pedro Queiroz de Andrade (10425822),
#              Júlia Andrade (10428513)
#
# Conteúdo: treina seis classificadores (TF-IDF ou embeddings, com regressão logística, SVM ou MLP)
#           para aprender as listas de categorias do treino, escolhe os botões e os cortes de cada um
#           por validação cruzada só no treino, mede os seis no gabarito humano, com a régua como
#           referência, e salva o melhor para a API.
#
# Alterações:
#   27/10/2026 - Bruna - notebook novo, com os classificadores medidos no gabarito humano
#   16/11/2026 - Bruna - versão final

A régua do notebook 07 dá categorias a qualquer atividade, mas só sabe medir a distância até os 16
centros, e a proposta da Parte 1 era treinar um classificador, um modelo que aprende com exemplos a dar
as categorias a uma atividade nova. Aqui treinamos seis, combinando duas formas de transformar o texto
em números com três classificadores:

| Representação do texto | Classificadores |
|---|---|
| TF-IDF | regressão logística, SVM, MLP |
| embeddings (Serafim) | regressão logística, SVM, MLP |

Os classificadores aprendem com as listas de categorias que a régua deu às 554 atividades de treino,
então a régua é a referência deles: ela aparece no placar para comparação, e não como mais um
concorrente. Tudo o que é escolha (o valor dos botões e o corte de cada categoria) sai de uma validação
cruzada dentro do treino, e o gabarito humano das 100 atividades de teste só aparece no fim, para medir
os seis classificadores e a régua.

In [2]:
# ====================================================================================================
#                      CARREGAMENTO DAS ATIVIDADES, DOS EMBEDDINGS E DO GABARITO
# ====================================================================================================
import pandas as pd                                           # leitura e manipulação das tabelas
import numpy as np                                            # contas com vetores e matrizes
import time                                                   # mede quanto tempo cada etapa custa
import warnings                                               # controla avisos repetidos das bibliotecas
from nltk.corpus import stopwords                             # palavras comuns do português
from sklearn.feature_extraction.text import TfidfVectorizer   # transforma texto em números pela contagem de palavras
from sklearn.multiclass import OneVsRestClassifier            # um classificador de sim ou não por categoria
from sklearn.linear_model import LogisticRegression           # regressão logística
from sklearn.svm import SVC                                   # SVM
from sklearn.neural_network import MLPClassifier              # rede neural com camadas escondidas
from sklearn.model_selection import KFold                     # divide o treino em partes para a validação cruzada
from sklearn.metrics import f1_score                          # nota que equilibra acertar e não deixar de fora

warnings.filterwarnings("ignore")   # a validação cruzada repete os mesmos avisos centenas de vezes

activities = pd.read_csv("../data/03_processed/07_activities_multilabel.csv", sep=";", encoding="utf-8-sig", keep_default_na=False)     # listas de categorias do notebook 07
activity_embeddings = np.load("../data/03_processed/03_embeddings.npy")                                                                 # vetores do notebook 03, na mesma ordem das linhas
gold_labels_table = pd.read_csv("../data/04_review/03_results/gold_labels.csv", sep=";", encoding="utf-8-sig", keep_default_na=False)   # o gabarito humano do notebook 08

category_names = sorted(column.replace("similarity_", "") for column in activities.columns if column.startswith("similarity_"))   # as 13 categorias, em ordem alfabética
is_train = (activities["split"] == "train").values                                                                                # a atividade é do treino?
train_activities = activities[is_train].reset_index(drop=True)                                                                    # as 554 de treino
gold_ids = gold_labels_table["activity_id"].values                                                                                # as 100 atividades do gabarito
gold_activities = activities.set_index("activity_id").loc[gold_ids].reset_index()                                                 # as mesmas 100, com o texto, na ordem do gabarito

print("Treino:", len(train_activities), "| Gabarito:", len(gold_activities), "| Categorias:", len(category_names))

Treino: 554 | Gabarito: 100 | Categorias: 13


In [3]:
# ----------------------------------------------------------------------------------------------------
#                  A RESPOSTA CERTA PARA APRENDER: UMA COLUNA DE 0 E 1 POR CATEGORIA
# ----------------------------------------------------------------------------------------------------
def split_labels(labels):
    return [label for label in labels.split(" | ") if label != ""]   # texto "A | B" vira a lista [A, B]

def label_matrix(list_of_labels):
    matrix = np.zeros((len(list_of_labels), len(category_names)), dtype=int)   # uma linha por atividade, uma coluna por categoria
    for row_number, labels in enumerate(list_of_labels):
        for label in split_labels(labels):
            matrix[row_number, category_names.index(label)] = 1                # 1 quando a atividade tem a categoria
    return matrix

train_targets = label_matrix(train_activities["labels"])                                                 # o que os classificadores aprendem: as listas da régua no treino
gold_targets = label_matrix(gold_activities.merge(gold_labels_table, on="activity_id")["gold_labels"])   # o gabarito humano no mesmo formato

pd.DataFrame({"train_activities_with_it": train_targets.sum(axis=0), "gold_activities_with_it": gold_targets.sum(axis=0)}, index=category_names)

,train_activities_with_it,gold_activities_with_it
Artes Visuais,80,11
Artesanato,49,14
Atividade Física e Bem-Estar,60,23
Cinema,52,7
Cultura e Identidade,62,19
Esporte,108,31
Infância e Brincadeiras,69,15
Literatura e Leitura,53,9
Música,37,18
Plantas e Alimentação,16,3


## A validação cruzada

Para escolher os botões sem olhar o teste, dividimos as 554 atividades de treino em 5 partes, e cada
classificador treina com 4 partes e responde a que ficou de fora, cinco vezes, até cada atividade de
treino ter sido respondida por um modelo que não a viu. É como estudar com 4 listas de exercício e
conferir na 5ª.

Cada classificador devolve uma probabilidade por categoria, e com as respostas das partes de fora também
escolhemos o corte de cada categoria, a partir de que probabilidade ela entra na lista. Testamos cortes
de 0,05 a 0,95 e ficamos com o que dá o maior F1 daquela categoria. Como a régua, todo classificador dá
pelo menos uma categoria: se nenhuma passar do corte, entra a de maior probabilidade.

Os botões testados são poucos: o C da regressão logística e do SVM (0,1, 1 e 10, e quanto maior, mais o
modelo tenta acertar cada exemplo do treino) e o tamanho da camada escondida da MLP (64 ou 128
neurônios). Em cada combinação, vence o valor com maior F1 médio entre as categorias.

In [4]:
# ====================================================================================================
#                                    AS PEÇAS DA VALIDAÇÃO CRUZADA
# ====================================================================================================
portuguese_stopwords = stopwords.words("portuguese")                             # mesma configuração do notebook 02
train_texts = train_activities["text"].values                                    # título e descrição das atividades de treino
train_embeddings = activity_embeddings[train_activities["activity_id"].values]   # vetores das atividades de treino

def features(representation, rows_to_learn, rows_to_answer, texts_to_answer=None, embeddings_to_answer=None):
    if representation == "TF-IDF":
        tfidf_vectorizer = TfidfVectorizer(stop_words=portuguese_stopwords)                                                            # vocabulário aprendido só com as linhas de aprender
        features_to_learn = tfidf_vectorizer.fit_transform(train_texts[rows_to_learn])                                                 # ajusta e transforma
        features_to_answer = tfidf_vectorizer.transform(train_texts[rows_to_answer] if texts_to_answer is None else texts_to_answer)   # só transforma
        return features_to_learn, features_to_answer, tfidf_vectorizer
    features_to_answer = train_embeddings[rows_to_answer] if embeddings_to_answer is None else embeddings_to_answer                    # embeddings não têm nada a ajustar
    return train_embeddings[rows_to_learn], features_to_answer, None

def build_classifier(classifier_name, setting):
    if classifier_name == "regressão logística":
        return OneVsRestClassifier(LogisticRegression(max_iter=1000, C=setting))                         # o código da aula, com o botão C
    if classifier_name == "SVM":
        return OneVsRestClassifier(SVC(kernel="linear", probability=True, C=setting, random_state=42))   # SVM linear que também devolve probabilidade
    return MLPClassifier(hidden_layer_sizes=(int(setting),), max_iter=500, random_state=42)              # uma camada escondida; já aceita várias categorias

threshold_grid = np.round(np.arange(0.05, 0.96, 0.05), 2)   # cortes testados: 0,05, 0,10 ... 0,95

def tune_thresholds(probabilities, targets):
    thresholds = []                                                                                                                                                                     # um corte por categoria
    for category_position in range(len(category_names)):
        f1_per_threshold = [f1_score(targets[:, category_position], (probabilities[:, category_position] >= threshold).astype(int), zero_division=0) for threshold in threshold_grid]   # F1 da categoria em cada corte
        thresholds.append(threshold_grid[int(np.argmax(f1_per_threshold))])                                                                                                             # fica o corte de maior F1
    return np.array(thresholds)

def apply_thresholds(probabilities, thresholds):
    predicted = (probabilities >= thresholds).astype(int)                                # entra toda categoria acima do próprio corte
    without_any = predicted.sum(axis=1) == 0                                             # atividades que ficaram sem nenhuma
    predicted[np.where(without_any)[0], probabilities[without_any].argmax(axis=1)] = 1   # essas recebem a de maior probabilidade
    return predicted

folds = list(KFold(n_splits=5, shuffle=True, random_state=42).split(train_texts))   # as mesmas 5 partes para todos os classificadores
print("Partes:", [len(rows_to_answer) for _, rows_to_answer in folds])

Partes: [111, 111, 111, 111, 110]


In [5]:
# ====================================================================================================
#                       VALIDAÇÃO CRUZADA DOS SEIS CLASSIFICADORES, SÓ NO TREINO
# ====================================================================================================
settings_to_try = {"regressão logística": [0.1, 1, 10], "SVM": [0.1, 1, 10], "MLP": [64, 128]}   # os botões testados em cada classificador

cross_validation_rows = []                                                                                                                                                                                                            # uma linha por combinação e valor de botão
out_of_fold_probabilities = {}                                                                                                                                                                                                        # (representação, classificador, botão) -> probabilidades das partes de fora
for representation in ["TF-IDF", "embeddings"]:
    for classifier_name, settings in settings_to_try.items():
        for setting in settings:
            time_before = time.time()                                                                                                                                                                                                 # marca o relógio antes
            probabilities = np.zeros(train_targets.shape)                                                                                                                                                                             # uma probabilidade por atividade de treino e categoria
            for rows_to_learn, rows_to_answer in folds:
                features_to_learn, features_to_answer, _ = features(representation, rows_to_learn, rows_to_answer)                                                                                                                    # texto vira número, só com as 4 partes
                classifier = build_classifier(classifier_name, setting).fit(features_to_learn, train_targets[rows_to_learn])                                                                                                          # aprende com 4 partes
                probabilities[rows_to_answer] = classifier.predict_proba(features_to_answer)                                                                                                                                          # responde a parte de fora
            thresholds = tune_thresholds(probabilities, train_targets)                                                                                                                                                                # um corte por categoria
            macro_f1 = f1_score(train_targets, apply_thresholds(probabilities, thresholds), average="macro", zero_division=0)                                                                                                         # F1 médio entre as categorias
            out_of_fold_probabilities[(representation, classifier_name, setting)] = probabilities                                                                                                                                     # guarda para usar depois
            cross_validation_rows.append({"representation": representation, "classifier": classifier_name, "setting": setting, "macro_f1_in_cross_validation": round(macro_f1, 4), "seconds": round(time.time() - time_before, 1)})   # guarda o resultado

cross_validation_table = pd.DataFrame(cross_validation_rows)   # lista vira tabela
cross_validation_table

,representation,classifier,setting,macro_f1_in_cross_validation,seconds
0,TF-IDF,regressão logística,0.1,0.2748,2.1
1,TF-IDF,regressão logística,1.0,0.6675,2.4
2,TF-IDF,regressão logística,10.0,0.7262,2.7
3,TF-IDF,SVM,0.1,0.7668,27.9
4,TF-IDF,SVM,1.0,0.7654,31.3
5,TF-IDF,SVM,10.0,0.7574,32.8
6,TF-IDF,MLP,64.0,0.7159,272.3
7,TF-IDF,MLP,128.0,0.7124,388.3
8,embeddings,regressão logística,0.1,0.8756,1.9
9,embeddings,regressão logística,1.0,0.8707,2.1


In [6]:
# ----------------------------------------------------------------------------------------------------
#                          O BOTÃO E OS CORTES ESCOLHIDOS EM CADA COMBINAÇÃO
# ----------------------------------------------------------------------------------------------------
best_rows = cross_validation_table.loc[cross_validation_table.groupby(["representation", "classifier"])["macro_f1_in_cross_validation"].idxmax()]   # o melhor botão de cada combinação

chosen = {}                                                                                                                                                                                     # (representação, classificador) -> botão e cortes escolhidos no treino
for _, row in best_rows.iterrows():
    probabilities = out_of_fold_probabilities[(row["representation"], row["classifier"], row["setting"])]                                                                                       # as respostas das partes de fora com esse botão
    chosen[(row["representation"], row["classifier"])] = {"setting": row["setting"], "thresholds": tune_thresholds(probabilities, train_targets), "out_of_fold_probabilities": probabilities}   # guarda o que foi escolhido

chosen_table = pd.DataFrame([{"representation": key[0], "classifier": key[1], "setting": value["setting"], "thresholds": " ".join(str(threshold) for threshold in value["thresholds"])} for key, value in chosen.items()])   # as escolhas numa tabela
chosen_table

,representation,classifier,setting,thresholds
0,TF-IDF,MLP,64.0,0.1 0.05 0.05 0.05 0.05 0.15 0.1 0.05 0.05 0.0...
1,TF-IDF,SVM,0.1,0.7 0.35 0.65 0.65 0.45 0.35 0.45 0.75 0.5 0.2...
2,TF-IDF,regressão logística,10.0,0.2 0.15 0.2 0.1 0.15 0.2 0.15 0.2 0.1 0.05 0....
3,embeddings,MLP,64.0,0.1 0.25 0.75 0.2 0.3 0.85 0.15 0.25 0.2 0.05 ...
4,embeddings,SVM,0.1,0.45 0.4 0.55 0.3 0.4 0.35 0.3 0.4 0.65 0.3 0....
5,embeddings,regressão logística,0.1,0.25 0.35 0.35 0.2 0.35 0.4 0.25 0.35 0.2 0.1 ...


**O que deu.** Com embeddings, os três classificadores ficaram entre 0,85 e 0,88 de F1 médio na
validação cruzada, e com TF-IDF, entre 0,72 e 0,77. No SVM com embeddings os três valores de C deram
exatamente o mesmo resultado, e ficou o primeiro. Três escolhas caíram na ponta da grade testada (C = 10
na regressão logística com TF-IDF, C = 0,1 na regressão logística com embeddings e no SVM com TF-IDF), e
não aumentamos a grade depois de ver isso.

Esses números medem o quanto cada classificador consegue imitar a régua em atividades de treino que ele
não viu, e ainda não dizem nada sobre acertar o que nós achamos. Antes de olhar o gabarito, anotamos as
escolhas e a regra do vencedor: vence quem tiver o maior F1 médio no gabarito, e ele vai para a API se
for viável rodar no computador do grupo.

## Os seis classificadores e a régua no gabarito humano

Cada combinação treina de novo com as 554 atividades de treino, com o botão e os cortes escolhidos
acima, e responde às 100 atividades do gabarito, enquanto a régua entra, como referência, com as listas
que já tinha dado no notebook 07. Medimos o F1 de cada categoria e a média entre as 13 (o F1 macro, que
trata categoria pequena e grande com o mesmo peso), o F1 somando todas as categorias juntas (o F1 micro,
em que as categorias grandes pesam mais), e também se a 1ª categoria prevista está entre as 1ªs válidas
do gabarito, que é a pergunta da Parte 2.

In [7]:
# ====================================================================================================
#                     TREINO FINAL E RESPOSTAS PARA AS 100 ATIVIDADES DO GABARITO
# ====================================================================================================
gold_texts = gold_activities["text"].values                                    # título e descrição das 100 atividades
gold_embeddings = activity_embeddings[gold_activities["activity_id"].values]   # os vetores delas
all_train_rows = np.arange(len(train_activities))                              # o treino inteiro

final_models = {}                                                                                                                                                                # (representação, classificador) -> tudo o que o modelo precisa para responder
gold_probabilities = {}                                                                                                                                                          # (representação, classificador) -> probabilidades nas 100 atividades
cost_rows = []                                                                                                                                                                   # tempo de cada um
for (representation, classifier_name), choice in chosen.items():
    time_before_training = time.time()                                                                                                                                           # marca o relógio antes do treino
    features_to_learn, features_to_answer, tfidf_vectorizer = features(representation, all_train_rows, None, texts_to_answer=gold_texts, embeddings_to_answer=gold_embeddings)   # texto vira número; o TF-IDF aprende só com o treino
    classifier = build_classifier(classifier_name, choice["setting"]).fit(features_to_learn, train_targets)                                                                      # aprende com o treino inteiro
    training_seconds = time.time() - time_before_training                                                                                                                        # quanto custou treinar

    time_before_answering = time.time()                                                                    # marca o relógio antes de responder
    if representation == "TF-IDF":
        features_to_answer = tfidf_vectorizer.transform(gold_texts)                                        # o tempo de responder inclui transformar o texto
    gold_probabilities[(representation, classifier_name)] = classifier.predict_proba(features_to_answer)   # uma probabilidade por categoria
    answering_seconds = time.time() - time_before_answering                                                # quanto custou responder às 100

    final_models[(representation, classifier_name)] = {"representation": representation, "classifier_name": classifier_name, "setting": choice["setting"], "classifier": classifier, "tfidf_vectorizer": tfidf_vectorizer, "thresholds": choice["thresholds"], "category_names": category_names}   # o pacote do modelo
    cost_rows.append({"representation": representation, "classifier": classifier_name, "training_seconds": round(training_seconds, 2), "answer_ms_per_activity_without_serafim": round(1000 * answering_seconds / len(gold_texts), 3)})                                                            # guarda o custo

pd.DataFrame(cost_rows)

,representation,classifier,training_seconds,answer_ms_per_activity_without_serafim
0,TF-IDF,MLP,51.74,0.110
1,TF-IDF,SVM,7.55,2.311
2,TF-IDF,regressão logística,0.65,0.160
3,embeddings,MLP,2.32,0.000
4,embeddings,SVM,4.17,0.440
5,embeddings,regressão logística,0.97,0.050


In [8]:
# ----------------------------------------------------------------------------------------------------
#                            QUANTO CUSTA O SERAFIM PARA UMA ATIVIDADE NOVA
# ----------------------------------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer   # o mesmo modelo de embeddings do notebook 03

time_before_loading = time.time()                                                             # marca o relógio antes
serafim_model = SentenceTransformer("PORTULAN/serafim-335m-portuguese-pt-sentence-encoder")   # carrega o modelo (uma vez só, quando a API sobe)
loading_seconds = time.time() - time_before_loading                                           # quanto custou carregar

new_embeddings = []                                                                          # um vetor por atividade
time_before_encoding = time.time()                                                           # marca o relógio antes
for gold_text in gold_texts[:20]:
    new_embeddings.append(serafim_model.encode([gold_text], show_progress_bar=False)[0])     # uma atividade por vez, como a API recebe
serafim_seconds_per_activity = (time.time() - time_before_encoding) / 20                     # custo médio por atividade, nas 20 primeiras do gabarito

print("Carregar o Serafim:", round(loading_seconds, 1), "s (uma vez) | gerar o vetor de uma atividade:", round(serafim_seconds_per_activity, 2), "s")
print("Os vetores gerados agora são os mesmos do notebook 03:", np.allclose(new_embeddings, gold_embeddings[:20], atol=1e-4))

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Carregar o Serafim: 4.5 s (uma vez) | gerar o vetor de uma atividade: 1.64 s
Os vetores gerados agora são os mesmos do notebook 03: True


In [9]:
# ====================================================================================================
#                                     O PLACAR NO GABARITO HUMANO
# ====================================================================================================
gold_first_categories = [split_labels(labels) for labels in gold_activities.merge(gold_labels_table, on="activity_id")["gold_first_categories"]]   # as 1ªs válidas de cada atividade

def ordered_labels(probabilities_of_one, predicted_of_one):
    positions = [position for position in np.argsort(-probabilities_of_one) if predicted_of_one[position] == 1]   # categorias previstas, da mais provável para a menos
    return [category_names[position] for position in positions]

predicted_lists = {}                                                                                                          # concorrente -> lista de categorias de cada uma das 100 atividades
competitor_keys = {}                                                                                                          # nome do concorrente -> (representação, classificador)
predicted_lists["régua (referência)"] = [split_labels(labels) for labels in gold_activities["labels"]]                        # as listas do notebook 07, das quais os classificadores aprenderam
for key in chosen:
    probabilities = gold_probabilities[key]                                                                                   # probabilidades do concorrente
    predicted = apply_thresholds(probabilities, final_models[key]["thresholds"])                                              # os cortes escolhidos no treino
    competitor_name = key[0] + " + " + key[1]                                                                                 # nome que aparece na tabela
    predicted_lists[competitor_name] = [ordered_labels(probabilities[row], predicted[row]) for row in range(len(gold_ids))]   # a lista em ordem de probabilidade
    competitor_keys[competitor_name] = key                                                                                    # para achar o modelo pelo nome

scoreboard_rows = []                                                                                                                                                                                                                                                                                                                         # uma linha por concorrente
f1_by_category = {}                                                                                                                                                                                                                                                                                                                          # concorrente -> F1 de cada categoria
for competitor, lists in predicted_lists.items():
    predicted_matrix = label_matrix([" | ".join(labels) for labels in lists])                                                                                                                                                                                                                                                                # previsões no formato de 0 e 1
    f1_by_category[competitor] = f1_score(gold_targets, predicted_matrix, average=None, zero_division=0)                                                                                                                                                                                                                                     # F1 de cada categoria
    first_hits = sum(1 for labels, gold_first in zip(lists, gold_first_categories) if labels and labels[0] in gold_first)                                                                                                                                                                                                                    # 1ª categoria entre as 1ªs válidas
    scoreboard_rows.append({"competitor": competitor, "macro_f1": round(f1_score(gold_targets, predicted_matrix, average="macro", zero_division=0), 3), "micro_f1": round(f1_score(gold_targets, predicted_matrix, average="micro", zero_division=0), 3), "first_category_hits": first_hits, "labels_given": int(predicted_matrix.sum())})   # as notas do concorrente

scoreboard = pd.DataFrame(scoreboard_rows).sort_values("macro_f1", ascending=False)   # do melhor para o pior no F1 médio
print("Categorias no gabarito:", int(gold_targets.sum()))
scoreboard

Categorias no gabarito: 194


,competitor,macro_f1,micro_f1,first_category_hits,labels_given
0,régua (referência),0.640,0.646,68,165
6,embeddings + regressão logística,0.630,0.649,63,148
5,embeddings + SVM,0.586,0.599,63,140
4,embeddings + MLP,0.565,0.599,62,133
2,TF-IDF + SVM,0.537,0.565,64,128
3,TF-IDF + regressão logística,0.528,0.572,57,138
1,TF-IDF + MLP,0.509,0.553,57,124


In [10]:
# ----------------------------------------------------------------------------------------------------
#                                         F1 DE CADA CATEGORIA
# ----------------------------------------------------------------------------------------------------
f1_by_category_table = pd.DataFrame(f1_by_category, index=category_names).round(2)   # linha: categoria; coluna: concorrente
f1_by_category_table["gold_activities_with_it"] = gold_targets.sum(axis=0)           # quantas atividades do gabarito têm a categoria
f1_by_category_table

,régua (referência),TF-IDF + MLP,TF-IDF + SVM,TF-IDF + regressão logística,embeddings + MLP,embeddings + SVM,embeddings + regressão logística,gold_activities_with_it
Artes Visuais,0.62,0.56,0.60,0.64,0.43,0.55,0.56,11
Artesanato,0.67,0.44,0.57,0.50,0.70,0.67,0.61,14
Atividade Física e Bem-Estar,0.32,0.28,0.15,0.21,0.28,0.21,0.28,23
Cinema,0.86,0.62,0.67,0.71,0.71,0.82,0.88,7
Cultura e Identidade,0.31,0.17,0.17,0.17,0.40,0.38,0.38,19
Esporte,0.91,0.83,0.83,0.83,0.82,0.82,0.88,31
Infância e Brincadeiras,0.53,0.43,0.61,0.54,0.57,0.50,0.60,15
Literatura e Leitura,0.64,0.78,0.75,0.71,0.70,0.67,0.67,9
Música,0.67,0.50,0.36,0.56,0.50,0.50,0.62,18
Plantas e Alimentação,0.86,0.50,0.40,0.50,0.67,0.67,0.67,3


In [11]:
# ----------------------------------------------------------------------------------------------------
#                 O MELHOR CLASSIFICADOR CONTRA A RÉGUA E CONTRA O SEGUNDO COLOCADO
# ----------------------------------------------------------------------------------------------------
reference_name = "régua (referência)"                                             # a régua não disputa: os classificadores aprenderam com ela
classifiers_scoreboard = scoreboard[scoreboard["competitor"] != reference_name]   # só os seis classificadores, do melhor para o pior
best_classifier = classifiers_scoreboard["competitor"].iloc[0]                    # o maior F1 macro entre os seis
second_classifier = classifiers_scoreboard["competitor"].iloc[1]                  # o segundo

def compare_by_category(name_a, name_b):
    difference = np.round(f1_by_category[name_a], 2) - np.round(f1_by_category[name_b], 2)   # F1 de cada categoria, como na tabela acima
    return {"melhor": int((difference > 0).sum()), "pior": int((difference < 0).sum()), "empate": int((difference == 0).sum())}

gaps_between_neighbors = (classifiers_scoreboard["macro_f1"] - classifiers_scoreboard["macro_f1"].shift(-1)).round(3)              # quanto cada classificador fica acima do seguinte
without_plants = [position for position, category_name in enumerate(category_names) if category_name != "Plantas e Alimentação"]   # todas as categorias menos a de 3 atividades no gabarito

print("Melhor classificador:", best_classifier, "| segundo:", second_classifier)
print("Distância de cada classificador para o seguinte:", dict(zip(classifiers_scoreboard["competitor"], gaps_between_neighbors)))
print(best_classifier, "contra a régua, por categoria:", compare_by_category(best_classifier, reference_name))
print("F1 macro sem Plantas e Alimentação | régua:", round(f1_by_category[reference_name][without_plants].mean(), 3), "|", best_classifier + ":", round(f1_by_category[best_classifier][without_plants].mean(), 3))
print(best_classifier, "contra", second_classifier + ", por categoria:", compare_by_category(best_classifier, second_classifier))

difference_to_second = pd.Series(f1_by_category[best_classifier] - f1_by_category[second_classifier], index=category_names).round(2)   # quanto o melhor fica acima do segundo em cada categoria
pd.DataFrame({"difference_to_second": difference_to_second, "gold_activities_with_it": gold_targets.sum(axis=0)}).sort_values("difference_to_second", ascending=False)

Melhor classificador: embeddings + regressão logística | segundo: embeddings + SVM
Distância de cada classificador para o seguinte: {'embeddings + regressão logística': 0.044, 'embeddings + SVM': 0.021, 'embeddings + MLP': 0.028, 'TF-IDF + SVM': 0.009, 'TF-IDF + regressão logística': 0.019, 'TF-IDF + MLP': nan}
embeddings + regressão logística contra a régua, por categoria: {'melhor': 6, 'pior': 6, 'empate': 1}
F1 macro sem Plantas e Alimentação | régua: 0.622 | embeddings + regressão logística: 0.627
embeddings + regressão logística contra embeddings + SVM, por categoria: {'melhor': 8, 'pior': 1, 'empate': 4}


,difference_to_second,gold_activities_with_it
Sociedade e Debates,0.17,21
Música,0.12,18
Infância e Brincadeiras,0.10,15
Atividade Física e Bem-Estar,0.07,23
Esporte,0.06,31
"Shows, Espetáculos e Performances",0.06,18
Cinema,0.05,7
Artes Visuais,0.01,11
Cultura e Identidade,0.00,19
Reciclagem e Sustentabilidade,0.00,5


**O que deu.** O melhor classificador foi o de embeddings com regressão logística, com F1 macro de
0,630, a 0,010 da régua (0,640), e no F1 micro ele fica até um pouco acima dela (0,649 contra 0,646).
Por categoria os dois empatam, já que cada um vai melhor em 6 das 13, e a vantagem da régua no F1 macro
vem de Plantas e Alimentação, que tem só 3 atividades no gabarito (0,86 contra 0,67): sem ela, o
classificador fica à frente (0,627 contra 0,622). Como os classificadores aprenderam com as listas que a
própria régua deu ao treino, chegar tão perto dela indica que esse classificador aprendeu bem o que ela
ensina.

Logo abaixo dele está o maior salto do placar. O segundo classificador, embeddings com SVM, fica 0,044
atrás (0,586), enquanto a distância entre os outros vizinhos de placar não passa de 0,028, nem mesmo na
passagem do último classificador com embeddings para o primeiro com TF-IDF. Esse salto não vem de uma
categoria pequena: a regressão logística vai melhor ou empata em 12 das 13 categorias, com as maiores
diferenças em categorias grandes, como Sociedade e Debates, Música e Infância e Brincadeiras.

Os classificadores com embeddings ficaram acima dos com TF-IDF (de 0,565 a 0,630, contra 0,509 a 0,537),
como esperávamos desde a Parte 2 e como já tinha acontecido na validação cruzada.

A régua acerta mais a 1ª categoria (68 das 100 atividades, contra 57 a 64 dos classificadores) e dá mais
categorias (165, contra 124 a 148), o que ajuda a cobrir o gabarito, que junta as listas de dois de nós.
Achamos que essa diferença vem do que vimos no notebook 07: os classificadores aprenderam com o treino,
em que 32% das atividades têm mais de uma categoria, enquanto no teste a régua dá mais de uma a 47%
delas.

Por categoria, o padrão se repete em todos: Esporte, Cinema e Shows vão bem, e Atividade Física e
Bem-Estar e Cultura e Identidade vão mal (F1 de 0,15 a 0,40). Essas duas categorias foram muito usadas
por nós como categoria seguinte, já que Cultura e Identidade apareceu como 1ª em só 3 das 19 atividades
em que está no gabarito, e Atividade Física em 9 das 23.

Com 100 atividades no gabarito, nenhuma dessas diferenças é grande o bastante para ser confirmada, e o
que elas indicam é a direção: para um classificador passar da régua, ele precisa aprender com rótulos
melhores que os dela, e para medir diferenças tão pequenas é preciso um gabarito humano maior.

In [12]:
# ====================================================================================================
#            AS PALAVRAS QUE MAIS PESAM EM CADA CATEGORIA (REGRESSÃO LOGÍSTICA COM TF-IDF)
# ====================================================================================================
logistic_tfidf = final_models[("TF-IDF", "regressão logística")]                # o modelo mais fácil de abrir por dentro
vocabulary_words = logistic_tfidf["tfidf_vectorizer"].get_feature_names_out()   # a palavra de cada coluna do TF-IDF

top_words_rows = []                                                                                               # uma linha por categoria
for category_position, category_name in enumerate(category_names):
    weights = logistic_tfidf["classifier"].estimators_[category_position].coef_[0]                                # o peso de cada palavra no classificador de sim ou não dessa categoria
    top_positions = np.argsort(-weights)[:8]                                                                      # as 8 palavras que mais empurram para o sim
    top_words_rows.append({"category": category_name, "top_words": ", ".join(vocabulary_words[top_positions])})   # guarda as palavras
pd.set_option("display.max_colwidth", 120)                                                                        # mostra a lista inteira de palavras
pd.DataFrame(top_words_rows)

,category,top_words
0,Artes Visuais,"artes, fotografia, criação, visuais, pintura, desenho, imagens, visual"
1,Artesanato,"oficina, cerâmica, modelagem, pré, técnicas, bumerangue, gravura, peças"
2,Atividade Física e Bem-Estar,"yoga, ginástica, movimento, corporal, movimentos, bem, dança, aula"
3,Cinema,"cinema, filmes, filme, indicativa, gênero, 11, direção, exibição"
4,Cultura e Identidade,"arte, memória, poesia, literatura, juventudes, cultural, negras, resistência"
5,Esporte,"esporte, mesa, modalidade, tênis, corrida, esportiva, recreação, treino"
6,Infância e Brincadeiras,"crianças, brincar, brincadeiras, espaço, lúdica, patinação, imaginação, bebês"
7,Literatura e Leitura,"histórias, livros, livro, leitura, literatura, língua, biblioteca, memórias"
8,Música,"musical, orquestra, música, instrumento, instrumentos, percussão, vocal, brasileira"
9,Plantas e Alimentação,"permacultura, plantas, independência, medicinais, horta, natureza, cuidados, sustentabilidade"


As palavras fazem sentido para quase todas as categorias e mostram o que o TF-IDF enxerga, que são
palavras do assunto, como "cerâmica", "yoga", "filmes" e "permacultura". Aparecem também marcas do jeito
de escrever de cada instituição: "indicativa" e "gênero" em Cinema vêm da ficha técnica dos filmes,
"fábrica" em Shows vem do nome das Fábricas de Cultura, "2026" em Reciclagem vem do nome da Virada
Sustentável 2026, e "bumerangue" em Artesanato vem de uma atividade isolada. Esse tipo de pista funciona
nesta base, mas pode não funcionar numa instituição nova.

In [13]:
# ====================================================================================================
#                                  O CLASSIFICADOR QUE VAI PARA A API
# ====================================================================================================
import joblib   # grava o classificador treinado num arquivo

api_key = competitor_keys[best_classifier]                                                                                                                     # (representação, classificador) do melhor dos seis
assert api_key[0] == "embeddings"                                                                                                                              # a API só gera o vetor do Serafim, então o classificador precisa ser de embeddings
api_model = final_models[api_key]                                                                                                                              # o pacote do modelo treinado com o treino inteiro
joblib.dump({"classifier": api_model["classifier"], "thresholds": api_model["thresholds"], "category_names": category_names}, "../models/classifier.joblib")   # o modelo, o corte de cada categoria e a ordem das categorias

answer_ms = [row["answer_ms_per_activity_without_serafim"] for row in cost_rows if (row["representation"], row["classifier"]) == api_key][0]   # quanto o classificador leva para responder, sem contar o Serafim
print("Classificador salvo em models/classifier.joblib:", best_classifier, "| botão:", api_model["setting"])
print("Custo para uma atividade nova: o vetor do Serafim (", round(serafim_seconds_per_activity, 2), "s) mais o classificador (", answer_ms, "ms)")

Classificador salvo em models/classifier.joblib: embeddings + regressão logística | botão: 0.1
Custo para uma atividade nova: o vetor do Serafim ( 1.64 s) mais o classificador ( 0.05 ms)


**O classificador da API.** Entre os seis, o de embeddings com regressão logística teve o maior F1
macro, e é ele que a API usa. Ele também é viável para o computador do grupo, porque, depois que o
Serafim gera o vetor da atividade (cerca de 1,6 s, medido aqui uma atividade por vez, que é como a API
vai receber), ele responde em menos de um milissegundo.

A régua não vai para a API, porque a proposta da Parte 1 era um classificador treinado e porque ela é a
fonte das listas que os classificadores aprenderam, servindo para medir o quanto eles aprenderam. Um
classificador também pode aprender de novo com as correções que fazemos quando a API pede revisão,
enquanto a régua fica presa aos centros dos grupos, e esse é o caminho para um dia passar dela.

O que a API ainda usa dos grupos são os 16 centros do notebook 07, que já estão em `models/`, para o
aviso de revisão do notebook 10 que confere se a atividade se parece com algum grupo.

O tempo do Serafim aqui (cerca de 1,6 s) é maior que o do notebook 03 (cerca de 0,45 s por atividade),
porque lá as 693 atividades foram transformadas de uma vez, e o modelo aproveita melhor o processador
com várias atividades juntas.